In [3]:
import numpy as np
import pandas as pd
from sklearn.model_selection import cross_val_score
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from scipy.stats import ttest_ind
from joblib import Parallel, delayed
import random
import time

# تنظیمات اولیه
random.seed(42)
np.random.seed(42)

# بارگذاری و پیش‌پردازش داده‌ها
data = pd.read_csv('Heart_disease_cleveland_new.csv')
X = data.drop('target', axis=1).values
y = data['target'].values
scaler = StandardScaler()
X = scaler.fit_transform(X)

# بازه پارامترهای C و gamma (در مقیاس لگاریتمی)
C_range = (-3, 3)  # log10(C) از 10^-3 تا 10^3
gamma_range = (-3, 3)  # log10(gamma) از 10^-3 تا 10^3

# حافظه برای تابع برازندگی
fitness_cache = {}

# تابع برازندگی
def fitness(individual):
    C_log, gamma_log = individual
    C = 10 ** C_log
    gamma = 10 ** gamma_log
    ind_tuple = (round(C_log, 4), round(gamma_log, 4))  # گرد کردن برای کاهش اندازه کش
    if ind_tuple in fitness_cache:
        return fitness_cache[ind_tuple]
    svm = SVC(C=C, gamma=gamma, kernel='rbf', random_state=42)
    accuracy = cross_val_score(svm, X, y, cv=5, scoring='accuracy').mean()
    fitness_cache[ind_tuple] = accuracy
    return accuracy

# جستجوی محلی با اندازه گام تطبیقی
def local_search(individual, n_iterations=10, initial_step_size=0.2):
    best_ind = individual.copy()
    best_fit = fitness(best_ind)
    step_size = initial_step_size

    for _ in range(n_iterations):
        new_ind = best_ind + np.random.normal(0, step_size, len(individual))
        new_ind = np.clip(new_ind, [C_range[0], gamma_range[0]], [C_range[1], gamma_range[1]])
        new_fit = fitness(new_ind)
        if new_fit > best_fit:
            best_ind = new_ind.copy()
            best_fit = new_fit
            step_size *= 0.9  # کاهش اندازه گام در صورت بهبود
        else:
            step_size *= 1.1  # افزایش اندازه گام در صورت عدم بهبود
        step_size = np.clip(step_size, 0.05, 0.5)  # محدود کردن اندازه گام
    return best_ind, best_fit

# الگوریتم ممتیک Lamarckian
def memetic_lamarckian(pop_size=20, n_generations=15, mutation_rate=0.15, crossover_rate=0.8, ls_iterations=5, patience=5):
    population = [np.random.uniform([C_range[0], gamma_range[0]], [C_range[1], gamma_range[1]], 2) for _ in range(pop_size)]
    best_fitness = -np.inf
    best_individual = None
    no_improvement = 0

    for generation in range(n_generations):
        fitness_scores = Parallel(n_jobs=4)(delayed(fitness)(ind) for ind in population)
        max_fitness = max(fitness_scores)
        mean_fitness = np.mean(fitness_scores)
        print(f"Lamarckian - نسل {generation+1}: بهترین برازندگی = {max_fitness:.4f}, میانگین برازندگی = {mean_fitness:.4f}")

        if max_fitness > best_fitness:
            best_fitness = max_fitness
            best_individual = population[np.argmax(fitness_scores)].copy()
            no_improvement = 0
        else:
            no_improvement += 1

        if no_improvement >= patience:
            print(f"توقف زودهنگام در نسل {generation+1}")
            break

        parents = []
        for _ in range(pop_size):
            tournament = random.sample(range(pop_size), 3)
            winner = tournament[np.argmax([fitness_scores[i] for i in tournament])]
            parents.append(population[winner].copy())

        new_population = []
        for i in range(0, pop_size, 2):
            if i + 1 < pop_size and random.random() < crossover_rate:
                parent1, parent2 = parents[i], parents[i + 1]
                alpha = random.random()
                child1 = alpha * parent1 + (1 - alpha) * parent2
                child2 = alpha * parent2 + (1 - alpha) * parent1
                new_population.extend([child1, child2])
            else:
                new_population.extend([parents[i].copy(), parents[i + 1].copy() if i + 1 < pop_size else parents[i].copy()])

        for ind in new_population:
            if random.random() < mutation_rate:
                ind += np.random.normal(0, 0.1, len(ind))
                ind = np.clip(ind, [C_range[0], gamma_range[0]], [C_range[1], gamma_range[1]])

        new_population = [local_search(ind, ls_iterations)[0] for ind in new_population]
        population = new_population[:pop_size - 1] + [best_individual.copy()]

    return best_individual, best_fitness

# الگوریتم ممتیک Baldwinian
def memetic_baldwinian(pop_size=20, n_generations=15, mutation_rate=0.15, crossover_rate=0.8, ls_iterations=5, patience=5):
    population = [np.random.uniform([C_range[0], gamma_range[0]], [C_range[1], gamma_range[1]], 2) for _ in range(pop_size)]
    best_fitness = -np.inf
    best_individual = None
    no_improvement = 0

    for generation in range(n_generations):
        fitness_scores = []
        ls_individuals = []
        for ind in population:
            ls_ind, ls_fit = local_search(ind, ls_iterations)
            fitness_scores.append(ls_fit)
            ls_individuals.append(ls_ind)

        max_fitness = max(fitness_scores)
        mean_fitness = np.mean(fitness_scores)
        print(f"Baldwinian - نسل {generation+1}: بهترین برازندگی = {max_fitness:.4f}, میانگین برازندگی = {mean_fitness:.4f}")

        if max_fitness > best_fitness:
            best_fitness = max_fitness
            best_individual = population[np.argmax(fitness_scores)].copy()
            no_improvement = 0
        else:
            no_improvement += 1

        if no_improvement >= patience:
            print(f"توقف زودهنگام در نسل {generation+1}")
            break

        parents = []
        for _ in range(pop_size):
            tournament = random.sample(range(pop_size), 3)
            winner = tournament[np.argmax([fitness_scores[i] for i in tournament])]
            parents.append(population[winner].copy())

        new_population = []
        for i in range(0, pop_size, 2):
            if i + 1 < pop_size and random.random() < crossover_rate:
                parent1, parent2 = parents[i], parents[i + 1]
                alpha = random.random()
                child1 = alpha * parent1 + (1 - alpha) * parent2
                child2 = alpha * parent2 + (1 - alpha) * parent1
                new_population.extend([child1, child2])
            else:
                new_population.extend([parents[i].copy(), parents[i + 1].copy() if i + 1 < pop_size else parents[i].copy()])

        for ind in new_population:
            if random.random() < mutation_rate:
                ind += np.random.normal(0, 0.1, len(ind))
                ind = np.clip(ind, [C_range[0], gamma_range[0]], [C_range[1], gamma_range[1]])

        population = new_population[:pop_size - 1] + [best_individual.copy()]

    return best_individual, best_fitness

# الگوریتم جستجوی محلی مستقل
def local_search_algorithm(n_iterations=100, initial_step_size=0.2):
    current_ind = np.random.uniform([C_range[0], gamma_range[0]], [C_range[1], gamma_range[1]], 2)
    best_ind = current_ind.copy()
    best_fit = fitness(best_ind)
    step_size = initial_step_size

    for i in range(n_iterations):
        new_ind = current_ind + np.random.normal(0, step_size, len(current_ind))
        new_ind = np.clip(new_ind, [C_range[0], gamma_range[0]], [C_range[1], gamma_range[1]])
        new_fit = fitness(new_ind)
        print(f"Local Search - تکرار {i+1}: برازندگی = {new_fit:.4f}")
        if new_fit > best_fit:
            best_ind = new_ind.copy()
            best_fit = new_fit
            current_ind = new_ind.copy()
            step_size *= 0.9
        else:
            step_size *= 1.1
        step_size = np.clip(step_size, 0.05, 0.5)

    return best_ind, best_fit

# اجرای الگوریتم‌ها
n_runs = 10
lamarckian_results = []
baldwinian_results = []
local_search_results = []

for run in range(n_runs):
    print(f"\n--- اجرای {run+1} ---")
    start_time = time.time()
    lam_ind, lam_fit = memetic_lamarckian()
    lam_time = time.time() - start_time
    lamarckian_results.append((lam_ind, lam_fit, lam_time))
    print(f"Lamarckian - زمان اجرا: {lam_time:.2f} ثانیه")

    start_time = time.time()
    bald_ind, bald_fit = memetic_baldwinian()
    bald_time = time.time() - start_time
    baldwinian_results.append((bald_ind, bald_fit, bald_time))
    print(f"Baldwinian - زمان اجرا: {bald_time:.2f} ثانیه")

    start_time = time.time()
    ls_ind, ls_fit = local_search_algorithm()
    ls_time = time.time() - start_time
    local_search_results.append((ls_ind, ls_fit, ls_time))
    print(f"Local Search - زمان اجرا: {ls_time:.2f} ثانیه")

# گزارش نتایج
def report_results(algo_name, results):
    fitnesses = [r[1] for r in results]
    params = [r[0] for r in results]
    times = [r[2] for r in results]
    mean_C = np.mean([10**p[0] for p in params])
    mean_gamma = np.mean([10**p[1] for p in params])
    print(f"\nنتایج {algo_name}:")
    print(f"میانگین دقت: {np.mean(fitnesses):.4f} ± {np.std(fitnesses):.4f}")
    print(f"بهترین دقت: {max(fitnesses):.4f}")
    print(f"میانگین C: {mean_C:.4f}, میانگین gamma: {mean_gamma:.4f}")
    print(f"میانگین زمان اجرا: {np.mean(times):.2f} ثانیه")

report_results("ممتیک Lamarckian", lamarckian_results)
report_results("ممتیک Baldwinian", baldwinian_results)
report_results("جستجوی محلی", local_search_results)

# مقایسه آماری
lam_fitnesses = [r[1] for r in lamarckian_results]
bald_fitnesses = [r[1] for r in baldwinian_results]
ls_fitnesses = [r[1] for r in local_search_results]

print("\nمقایسه آماری (t-test):")
t_stat_lb, p_value_lb = ttest_ind(lam_fitnesses, bald_fitnesses)
print(f"Lamarckian vs Baldwinian: t-statistic = {t_stat_lb:.4f}, p-value = {p_value_lb:.4f}")
t_stat_ll, p_value_ll = ttest_ind(lam_fitnesses, ls_fitnesses)
print(f"Lamarckian vs Local Search: t-statistic = {t_stat_ll:.4f}, p-value = {p_value_ll:.4f}")
t_stat_bl, p_value_bl = ttest_ind(bald_fitnesses, ls_fitnesses)
print(f"Baldwinian vs Local Search: t-statistic = {t_stat_bl:.4f}, p-value = {p_value_bl:.4f}")


--- اجرای 1 ---
Lamarckian - نسل 1: بهترین برازندگی = 0.8480, میانگین برازندگی = 0.6239
Lamarckian - نسل 2: بهترین برازندگی = 0.8514, میانگین برازندگی = 0.7424
Lamarckian - نسل 3: بهترین برازندگی = 0.8514, میانگین برازندگی = 0.8457
Lamarckian - نسل 4: بهترین برازندگی = 0.8514, میانگین برازندگی = 0.8481
Lamarckian - نسل 5: بهترین برازندگی = 0.8514, میانگین برازندگی = 0.8505
Lamarckian - نسل 6: بهترین برازندگی = 0.8514, میانگین برازندگی = 0.8510
Lamarckian - نسل 7: بهترین برازندگی = 0.8514, میانگین برازندگی = 0.8504
توقف زودهنگام در نسل 7
Lamarckian - زمان اجرا: 25.00 ثانیه
Baldwinian - نسل 1: بهترین برازندگی = 0.8446, میانگین برازندگی = 0.6267
Baldwinian - نسل 2: بهترین برازندگی = 0.8447, میانگین برازندگی = 0.7330
Baldwinian - نسل 3: بهترین برازندگی = 0.8480, میانگین برازندگی = 0.8056
Baldwinian - نسل 4: بهترین برازندگی = 0.8479, میانگین برازندگی = 0.8384
Baldwinian - نسل 5: بهترین برازندگی = 0.8514, میانگین برازندگی = 0.8426
Baldwinian - نسل 6: بهترین برازندگی = 0.8514, میانگین برازند

/usr/local/lib/python3.11/dist-packages/scipy/stats/_axis_nan_policy.py:586: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)
